# Oral-microbiome analysis of original MetaCyc pathways

This notebook is an independent downstream analysis of five **precomputed** molecular patterns. It does not fit trajectories, import the clustering module, or alter cluster membership.

Three inputs have separate, non-interchangeable roles. `01-interim_data/omics_data/oral_microbiome_with_metadata.parquet` supplies **raw, un-imputed MetaPhlAn species abundance** to define the candidate reference cohort and audit current-cohort prevalence of the already clustered taxa; this audit does not alter frozen P1–P5 membership. `pathway_abundance_pathway_level.arrow` supplies **community-level MetaCyc pathway abundance** for pathway preprocessing. `pathway_abundance_microbe_level.arrow` supplies **species-stratified RPK** for contributor assignment. The former `070-oral_microbiome_imputed.parquet` is not used.

The three matrices are matched by an explicit sample key before preprocessing. The final cohort is their complete-case intersection: a sample absent from any matrix is excluded from all three, and missing HUMAnN rows are never imputed as zero. Row order and equal row counts are never treated as proof of sample identity.

The primary result is a **quantitative descriptive five-pattern landscape** of original HUMAnN/MetaCyc pathway IDs. The exact number of fixed oral taxa is read dynamically from the precomputed cluster table and is never hard-coded. Main-figure pathways pass prespecified community detection and stable taxon-contribution rules and are ranked across the complete frozen cluster universe by contributor count, detection support and mean RPK; pattern-specific P values are never used for figure entry. No project-specific functional-theme dictionary or regex-defined biological module is used.

Taxon-set carriage enrichment is a separate **exploratory supplementary analysis**: P1–P5 are compared with the other four patterns using one-sided Fisher exact tests, global BH-FDR and a strict Bonferroni sensitivity analysis. Neither correction controls the primary heatmap. Pattern-header n values denote oral microbial taxa, whereas the complete-case sample count supports prevalence and all-sample RPK summaries. Microbial functional potential must not be interpreted as host-pathway activation.

## 1. Configure paths and prespecified thresholds

The default raw species input is `02-papers/01-interim_data/omics_data/oral_microbiome_with_metadata.parquet`. The two HUMAnN defaults are the separate `pathway_abundance_pathway_level.arrow` and `pathway_abundance_microbe_level.arrow` files shown in the available dataset. The default shared sample key is `participant_id`; set `ORAL_SAMPLE_KEY_COLUMNS=participant_id,research_stage` only when that composite biospecimen key is present and unique in all three matrices. Set path or threshold environment variables only when the remote location differs. Outputs use `oral_metacyc_pathway_analysis`, isolated from legacy module-based results.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import uuid

import pandas as pd
from IPython.display import Image, display

RESULT2_DIR = Path(os.environ.get('RESULT_DIR', Path.cwd())).resolve()  # directory of this notebook
CLUSTER_METHOD = os.environ.get('MOLECULE_CLUSTER_METHOD', 'agglomerative')
MOLECULE_CLUSTERS = (
    RESULT2_DIR / 'outputs' / 'molecule_clustering'
    / f'molecule_cluster_assignments_{CLUSTER_METHOD}.csv'
)
ORAL_ANALYSIS_SCRIPT = (
    RESULT2_DIR / 'analyze_oral_cluster_pathways.py'
)
ORAL_ABUNDANCE_TABLE = Path(os.environ.get(
    'ORAL_MICROBIOME_WITH_METADATA',
    str(
        RESULT2_DIR.parent / '01-interim_data' / 'omics_data'
        / 'oral_microbiome_with_metadata.parquet'
    ),
))
ORAL_COMMUNITY_PATHWAY_ARROW = Path(os.environ.get(
    'ORAL_COMMUNITY_PATHWAY_ARROW',
    '/home/ec2-user/studies/hpp_datasets/oral_microbiome/aggregated/'
    'humann_aggregated/pathway_abundance_pathway_level.arrow',
))
ORAL_MICROBE_PATHWAY_ARROW = Path(os.environ.get(
    'ORAL_MICROBE_PATHWAY_ARROW',
    os.environ.get(
        'ORAL_STRATIFIED_ARROW',
        '/home/ec2-user/studies/hpp_datasets/oral_microbiome/aggregated/'
        'humann_aggregated/pathway_abundance_microbe_level.arrow',
    ),
))
ORAL_SAMPLE_KEY_COLUMNS = os.environ.get(
    'ORAL_SAMPLE_KEY_COLUMNS', 'participant_id'
).strip()
if not ORAL_SAMPLE_KEY_COLUMNS:
    raise ValueError('ORAL_SAMPLE_KEY_COLUMNS must not be empty')
ORAL_PATHWAY_OUTPUT_DIR = (
    RESULT2_DIR / 'outputs' / 'oral_metacyc_pathway_analysis'
)
ORAL_PATHWAY_FIGURE_DIR = (
    RESULT2_DIR / 'figures' / 'oral_metacyc_pathway_analysis'
)

ORAL_MIN_SPECIES_DETECTED_SAMPLES = int(os.environ.get(
    'ORAL_MIN_SPECIES_DETECTED_SAMPLES', '500'
))
ORAL_MIN_PATHWAY_DETECTED_SAMPLES = int(os.environ.get(
    'ORAL_MIN_PATHWAY_DETECTED_SAMPLES', '500'
))
ORAL_MIN_CONTRIBUTION_SAMPLES = int(os.environ.get(
    'ORAL_MIN_CONTRIBUTION_SAMPLES', '500'
))
ORAL_PATHWAY_DETECTION_PERCENTILE = float(os.environ.get(
    'ORAL_PATHWAY_DETECTION_PERCENTILE', '1'
))
ORAL_MIN_CARRIER_TAXA = int(os.environ.get('ORAL_MIN_CARRIER_TAXA', '3'))
ORAL_MIN_NONCARRIER_TAXA = int(os.environ.get('ORAL_MIN_NONCARRIER_TAXA', '3'))
ORAL_MIN_FIGURE_CARRIERS = int(os.environ.get('ORAL_MIN_FIGURE_CARRIERS', '2'))
ORAL_MIN_OVERALL_MAPPING_RATE = float(os.environ.get('ORAL_MIN_OVERALL_MAPPING_RATE', '0.50'))
ORAL_MIN_PATTERN_MAPPING_RATE = float(os.environ.get('ORAL_MIN_PATTERN_MAPPING_RATE', '0.25'))
ORAL_ALPHA = float(os.environ.get('ORAL_ALPHA', '0.05'))
ORAL_MAX_PATHWAYS = int(os.environ.get('ORAL_MAX_PATHWAYS', '8'))
ORAL_REPRESENTATIVE_TAXA = int(os.environ.get('ORAL_REPRESENTATIVE_TAXA', '2'))
ORAL_PREVIEW_DPI = int(os.environ.get('ORAL_PREVIEW_DPI', '450'))
ORAL_SCOPE_EXCLUSION_REGEX = os.environ.get(
    'ORAL_SCOPE_EXCLUSION_REGEX',
    '',
)
ORAL_RUN_ID = uuid.uuid4().hex

print(f'Result directory: {RESULT2_DIR}')
print(f'Precomputed cluster input: {MOLECULE_CLUSTERS}')
print(f'Raw oral species abundance: {ORAL_ABUNDANCE_TABLE}')
print(f'HUMAnN community pathway table: {ORAL_COMMUNITY_PATHWAY_ARROW}')
print(f'HUMAnN microbe-level pathway table: {ORAL_MICROBE_PATHWAY_ARROW}')
print(f'Sample-key columns: {ORAL_SAMPLE_KEY_COLUMNS}')
print(f'Current run ID: {ORAL_RUN_ID}')
print('Clustering execution in this notebook: disabled')
print('Functional-theme dictionary dependency: disabled')
print('070-oral_microbiome_imputed.parquet dependency: disabled')
print('Legacy module output directory: excluded')

## 2. Audit the existing five-pattern assignment

This cell only reads the exported assignment table. It verifies that all five oral-taxon patterns are present and reports taxa counts; it does not recompute labels.

In [ ]:
oral_cluster_input_ready = False
oral_cluster_table = None
if not MOLECULE_CLUSTERS.exists():
    print(f'Precomputed cluster result is missing: {MOLECULE_CLUSTERS}')
else:
    cluster_table = pd.read_csv(MOLECULE_CLUSTERS)
    required_cluster_columns = {'feature', 'omics', 'pattern_id', 'pattern'}
    missing_cluster_columns = sorted(
        required_cluster_columns - set(cluster_table.columns)
    )
    if missing_cluster_columns:
        print('Cluster table is missing columns: ' + ', '.join(missing_cluster_columns))
    else:
        oral_cluster_table = cluster_table.loc[
            cluster_table['omics'].astype(str) == 'Oral microbiome'
        ].copy()
        oral_cluster_table['pattern_id'] = pd.to_numeric(
            oral_cluster_table['pattern_id'], errors='coerce'
        )
        oral_cluster_table = oral_cluster_table.loc[
            oral_cluster_table['pattern_id'].notna()
        ].copy()
        oral_cluster_table['pattern_id'] = oral_cluster_table['pattern_id'].astype(int)
        oral_pattern_counts = (
            oral_cluster_table['pattern_id'].value_counts()
            .reindex([1, 2, 3, 4, 5], fill_value=0)
            .rename_axis('pattern_id')
            .rename('oral_taxa_n')
            .reset_index()
        )
        display(oral_pattern_counts)
        missing_patterns = oral_pattern_counts.loc[
            oral_pattern_counts['oral_taxa_n'] == 0, 'pattern_id'
        ].tolist()
        oral_cluster_input_ready = (
            len(oral_cluster_table) > 0
            and not missing_patterns
            and not oral_cluster_table['feature'].duplicated().any()
        )
        print(f'Oral taxa in existing cluster result: {len(oral_cluster_table)}')
        print(f'All five oral patterns present: {not missing_patterns}')
        print(f'Oral feature identifiers unique: {not oral_cluster_table["feature"].duplicated().any()}')

print(f'Existing cluster input ready: {oral_cluster_input_ready}')
print('No clustering function was imported or executed.')

## 3. Run pathway-level analysis

Before preprocessing, the oral Parquet and both HUMAnN Arrow matrices are aligned by the configured sample key. The final analysis uses their complete-case intersection; samples missing from any source and source-only rows are excluded synchronously before every prevalence audit, threshold, normalization and contribution calculation. Missing sample rows are not converted to biological zeros. The raw oral prevalence threshold is an audit of the frozen cluster members and does not remove them a second time. Pathway preprocessing follows the supplied method where the available data support it: exclude HUMAnN special features from threshold estimation; calculate one global first percentile from all positive, non-zero community pathway values; replace zeros/nulls with half that threshold for normalization only; sum-normalize within samples, scale to PPM and log10-transform; and require raw community abundance at or above the threshold in at least 500 samples. A pathway–taxon edge independently requires raw species-stratified RPK > 0 in at least 500 samples.

Because this analysis has no metabolic-health outcome, the screenshot's age/sex-adjusted OLS and outcome-outlier processing are not applicable and are not fabricated. The primary landscape ranks descriptively eligible pathways by overall stable contributor-taxon count, median edge-detection support and total all-sample mean RPK, without a P-value gate or cluster-specific quota. One-sided Fisher tests, global BH-FDR and Bonferroni sensitivity values are written for every eligible pathway × pattern comparison in a separate exploratory supplementary table.

In [ ]:
oral_pathway_run_performed = False
oral_pathway_status = 'not started'
oral_required_inputs = {
    'precomputed five-pattern table': MOLECULE_CLUSTERS,
    'raw oral species-abundance Parquet': ORAL_ABUNDANCE_TABLE,
    'community-level HUMAnN pathway table': ORAL_COMMUNITY_PATHWAY_ARROW,
    'microbe-level HUMAnN pathway table': ORAL_MICROBE_PATHWAY_ARROW,
    'pathway-level analysis script': ORAL_ANALYSIS_SCRIPT,
}
oral_missing_inputs = {
    name: path for name, path in oral_required_inputs.items() if not path.exists()
}

if not oral_cluster_input_ready:
    oral_pathway_status = 'skipped: existing five-pattern result is invalid'
    print(oral_pathway_status)
elif oral_missing_inputs:
    oral_pathway_status = 'skipped: required inputs unavailable'
    print('Oral MetaCyc pathway analysis skipped; missing inputs:')
    for name, path in oral_missing_inputs.items():
        print(f'  {name}: {path}')
else:
    oral_pathway_command = [
        sys.executable, str(ORAL_ANALYSIS_SCRIPT),
        '--oral-abundance-table', str(ORAL_ABUNDANCE_TABLE),
        '--community-pathway-table', str(ORAL_COMMUNITY_PATHWAY_ARROW),
        '--microbe-pathway-table', str(ORAL_MICROBE_PATHWAY_ARROW),
        '--sample-key-columns', ORAL_SAMPLE_KEY_COLUMNS,
        '--molecule-clusters', str(MOLECULE_CLUSTERS),
        '--output-dir', str(ORAL_PATHWAY_OUTPUT_DIR),
        '--figure-dir', str(ORAL_PATHWAY_FIGURE_DIR),
        '--run-id', ORAL_RUN_ID,
        '--min-species-detected-samples', str(
            ORAL_MIN_SPECIES_DETECTED_SAMPLES
        ),
        '--min-pathway-detected-samples', str(
            ORAL_MIN_PATHWAY_DETECTED_SAMPLES
        ),
        '--min-contribution-samples', str(
            ORAL_MIN_CONTRIBUTION_SAMPLES
        ),
        '--pathway-detection-percentile', str(
            ORAL_PATHWAY_DETECTION_PERCENTILE
        ),
        '--min-carrier-taxa', str(ORAL_MIN_CARRIER_TAXA),
        '--min-noncarrier-taxa', str(ORAL_MIN_NONCARRIER_TAXA),
        '--min-figure-carriers', str(ORAL_MIN_FIGURE_CARRIERS),
        '--min-overall-mapping-rate', str(ORAL_MIN_OVERALL_MAPPING_RATE),
        '--min-pattern-mapping-rate', str(ORAL_MIN_PATTERN_MAPPING_RATE),
        '--alpha', str(ORAL_ALPHA),
        '--max-pathways', str(ORAL_MAX_PATHWAYS),
        '--representative-taxa', str(ORAL_REPRESENTATIVE_TAXA),
        '--preview-dpi', str(ORAL_PREVIEW_DPI),
        '--scope-exclusion-regex', ORAL_SCOPE_EXCLUSION_REGEX,
    ]
    oral_pathway_run_performed = True
    oral_pathway_run = subprocess.run(
        oral_pathway_command,
        text=True,
        capture_output=True,
    )
    if oral_pathway_run.stdout:
        print(oral_pathway_run.stdout)
    if oral_pathway_run.stderr:
        print(oral_pathway_run.stderr, file=sys.stderr)
    if oral_pathway_run.returncode != 0:
        raise RuntimeError(
            'Oral MetaCyc pathway analysis failed with return code '
            f'{oral_pathway_run.returncode}; see the child traceback above.'
        )
    oral_pathway_status = 'script finished; inspect current-run manifest'

print(f'Oral MetaCyc pathway run status: {oral_pathway_status}')

## 4. Inspect provenance, descriptive selection and exploratory statistics

The manifest is accepted only when its run ID matches this notebook execution. This prevents a skipped or failed run from displaying stale results produced by the previous module-based method.

In [ ]:
oral_pathway_manifest_file = (
    ORAL_PATHWAY_OUTPUT_DIR / 'oral_metacyc_pathway_analysis_manifest.json'
)
oral_pathway_manifest = None
if not oral_pathway_run_performed:
    print('No pathway run was performed in this notebook session; stale manifests are not displayed.')
elif not oral_pathway_manifest_file.exists():
    raise FileNotFoundError(
        f'The analysis finished without a manifest: {oral_pathway_manifest_file}'
    )
else:
    candidate_manifest = json.loads(
        oral_pathway_manifest_file.read_text(encoding='utf-8')
    )
    if candidate_manifest.get('run_id') != ORAL_RUN_ID:
        raise RuntimeError(
            'Manifest run ID does not match this notebook execution; refusing stale output.'
        )
    oral_pathway_manifest = candidate_manifest
    cluster_audit = oral_pathway_manifest.get('cluster_universe', {})
    sample_alignment = oral_pathway_manifest.get('sample_alignment', {})
    coverage_audit = oral_pathway_manifest.get('species_mapping_coverage', {})
    scope_audit = oral_pathway_manifest.get('pathway_scope_filter', {})
    pathway_preprocessing = oral_pathway_manifest.get(
        'community_pathway_preprocessing', {}
    )
    filtering_audit = oral_pathway_manifest.get('independent_filtering', {})
    test_audit = oral_pathway_manifest.get('exploratory_statistical_tests', {})
    selection_audit = oral_pathway_manifest.get('figure_selection', {})
    display(pd.DataFrame({
        'analysis_version': [oral_pathway_manifest.get('analysis_version')],
        'status': [oral_pathway_manifest.get('status')],
        'sample_key_columns': [sample_alignment.get('sample_key_columns')],
        'oral_reference_samples': [sample_alignment.get('oral_reference_samples')],
        'community_input_samples': [sample_alignment.get('community_input_samples')],
        'microbe_input_samples': [sample_alignment.get('microbe_input_samples')],
        'aligned_analysis_samples': [sample_alignment.get('analysis_samples')],
        'oral_reference_samples_excluded_complete_case': [sample_alignment.get(
            'oral_reference_samples_excluded_complete_case'
        )],
        'complete_case_retention_fraction': [sample_alignment.get(
            'complete_case_retention_fraction'
        )],
        'oral_samples_missing_from_community': [sample_alignment.get(
            'oral_samples_missing_from_community'
        )],
        'oral_samples_missing_from_microbe': [sample_alignment.get(
            'oral_samples_missing_from_microbe'
        )],
        'community_extra_samples_excluded': [sample_alignment.get(
            'community_extra_samples_excluded'
        )],
        'microbe_extra_samples_excluded': [sample_alignment.get(
            'microbe_extra_samples_excluded'
        )],
        'fixed_cluster_taxa': [cluster_audit.get('universe_n')],
        'taxa_meeting_reference_prevalence_audit': [cluster_audit.get(
            'taxa_meeting_reference_prevalence_audit'
        )],
        'P1–P5 taxa': [cluster_audit.get('pattern_counts')],
        'taxonomy_mapping_rate': [coverage_audit.get('overall_rate')],
        'pathway_detection_threshold': [pathway_preprocessing.get(
            'detection_threshold'
        )],
        'community_pathways_after_prevalence': [pathway_preprocessing.get(
            'retained_pathways'
        )],
        'scope-excluded_pathways': [scope_audit.get('pathways_excluded')],
        'descriptive_pathways': [filtering_audit.get('descriptive_pathways')],
        'exploratory_pathways_in_global_test_family': [filtering_audit.get(
            'exploratory_pathways_in_global_test_family'
        )],
        'exploratory_pathway×pattern_tests': [test_audit.get('global_test_count')],
        'BH-FDR-positive_pathways_exploratory': [test_audit.get(
            'positive_pathways_global_bh_fdr'
        )],
        'Bonferroni-positive-pathways_sensitivity': [test_audit.get(
            'positive_pathways_global_bonferroni_sensitivity'
        )],
        'primary_figure_uses_p_value_gate': [selection_audit.get(
            'p_value_gate_used'
        )],
        'displayed_pathways': [selection_audit.get('displayed_pathways')],
        'skip_reason': [oral_pathway_manifest.get('skip_reason')],
    }))
    excluded_inputs = oral_pathway_manifest['analysis_method']['explicitly_not_used']
    print('Inputs and interpretations explicitly excluded:')
    for item in excluded_inputs:
        print(f'  - {item}')
    if 'functional_theme_dictionary.tsv' not in excluded_inputs:
        raise RuntimeError('The current manifest does not document dictionary exclusion.')
    manifest_oral_input = Path(
        oral_pathway_manifest['inputs']['oral_abundance_table']['path']
    )
    if manifest_oral_input.resolve() != ORAL_ABUNDANCE_TABLE.resolve():
        raise RuntimeError('Manifest raw oral input differs from this notebook run.')
    if '070-oral_microbiome_imputed.parquet' in str(manifest_oral_input):
        raise RuntimeError('The prohibited imputed 070 table entered the run.')
    manifest_community_input = Path(
        oral_pathway_manifest['inputs']['community_pathway_table']['path']
    )
    manifest_microbe_input = Path(
        oral_pathway_manifest['inputs']['microbe_pathway_table']['path']
    )
    if manifest_community_input.resolve() != ORAL_COMMUNITY_PATHWAY_ARROW.resolve():
        raise RuntimeError('Manifest community-pathway input differs from this run.')
    if manifest_microbe_input.resolve() != ORAL_MICROBE_PATHWAY_ARROW.resolve():
        raise RuntimeError('Manifest microbe-pathway input differs from this run.')
    print('Primary interpretation: descriptive taxon-contribution landscape, not enrichment or host activation.')
    print('Exploratory Fisher/BH/Bonferroni results are supplementary and do not gate the main figure.')

## 5. Preview current-run pathway results

The primary figure is a descriptive five-pattern heatmap with stable contributor support; it contains no P values or significance gate. PDF and SVG are the authoritative editable main-figure files. PNG is an optional preview. The complete Fisher/BH/Bonferroni table is previewed separately and retained as exploratory supplementary output. TIFF is intentionally not generated for the main figure.

In [ ]:
if oral_pathway_manifest is None:
    print('Nothing from the current run to preview.')
elif oral_pathway_manifest.get('status') == 'completed':
    output_files = oral_pathway_manifest.get('outputs', {})
    selection_file = Path(
        output_files['metacyc_pathway_descriptive_figure_selection']
    )
    selection = pd.read_csv(selection_file)
    display(selection[[
        'pathway_id', 'pathway_name', 'dominant_pattern',
        'overall_stable_contributor_taxa_n', 'retained_taxa_n',
        'overall_stable_detection_samples_median',
        'total_mean_rpk_all_taxa', 'mean_rpk_per_retained_taxon',
        'community_detected_samples', 'community_mean_log10_ppm',
        'representative_taxa',
    ]])
    exploratory_file = Path(
        output_files['metacyc_pathway_enrichment_exploratory_all']
    )
    exploratory = pd.read_csv(exploratory_file)
    print('Exploratory supplementary enrichment (top 20 by global BH-FDR):')
    display(exploratory.sort_values(
        ['q_bh_global', 'p_fisher_greater', 'log2_odds_ratio_ha'],
        ascending=[True, True, False],
        kind='mergesort',
    )[[
        'pathway_id', 'pathway_name', 'pattern_id',
        'foreground_with', 'foreground_n',
        'log2_odds_ratio_ha', 'ci95_low_log2_ha',
        'ci95_high_log2_ha', 'p_fisher_greater',
        'q_bh_global', 'p_bonferroni_global',
        'exploratory_positive_bh',
        'strict_positive_bonferroni_sensitivity',
    ]].head(20))
    figure_files = oral_pathway_manifest.get('figure', {}).get('files', {})
    if figure_files.get('png') and Path(figure_files['png']).exists():
        display(Image(filename=figure_files['png'], width=1500))
    else:
        print('Optional PNG preview is unavailable; use the vector files below.')
    print(f"Authoritative PDF: {figure_files.get('pdf')}")
    print(f"Editable SVG: {figure_files.get('svg')}")
elif oral_pathway_manifest.get('status') == 'skipped':
    print(f"Figure skipped: {oral_pathway_manifest.get('skip_reason')}")
else:
    failure = oral_pathway_manifest.get('failure', {})
    print(f"Run status: {oral_pathway_manifest.get('status')}")
    print(f"Failure: {failure.get('type')}: {failure.get('message')}")

## Interpretation checklist

1. Report taxa n for P1–P5; these are not participant sample sizes.
2. Report the sample-key columns, all three input sample counts, source-only rows, oral-reference samples excluded by complete-case analysis, retention fraction, and final analysis n.
3. Report the global positive-value percentile and resulting numeric pathway threshold, zero/null replacement, PPM/log10 transformation, and all three ≥500-sample filters.
4. Treat representative taxa, stable-support bars and heatmap abundance as descriptive source-data annotations; main-figure pathway selection must not use P or q values.
5. Do not call any result a host pathway, pathway activation, or causal host–microbe mechanism.
6. Treat the one-sided Fisher tests and global BH-FDR as exploratory supplementary analyses; Bonferroni is a strict sensitivity analysis. Report effect sizes, 95% CIs, raw P, BH q and Bonferroni values without allowing them to gate the descriptive figure.
7. Inspect the complete sample-alignment, species-preprocessing, community-pathway preprocessing, scope and role-specific filtering audits. Displayed pathways are a prespecified descriptive subset ranked by global taxon support, detection support and mean RPK, not a subset of significant results.
8. Do not reinterpret a missing HUMAnN sample row as zero abundance; sample-level missingness is handled only by complete-case exclusion.
9. If the exploratory test family is empty but descriptively eligible pathways remain, preserve the descriptive main figure and report zero exploratory tests; skip only when the required inputs or descriptive eligibility criteria fail.